# KNN - Customer Churn Classification

This notebook is designed for classroom demonstration with clear comments and simple steps.

## Objective

In this notebook, we will demonstrate a basic **classification** machine learning flow.

## Basic Steps

1. Read the dataset  
2. Perform basic EDA  
3. Prepare input features and target  
4. Convert categorical columns into numeric columns  
5. Split data into training and testing sets  
6. Train the model  
7. Predict  
8. Evaluate using suitable metrics  
9. Save the trained model using pickle  

Dataset used: `customer_churn_classification_1000.csv`


In [ ]:
# ============================================================
# Import required libraries
# ============================================================

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Libraries imported successfully.")

In [ ]:
# Read dataset
df = pd.read_csv("../datasets/customer_churn_classification_1000.csv")
df.head()

In [ ]:
# Basic EDA
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["churn"].value_counts())
df.describe()

In [ ]:
# Prepare X and y
X = df.drop("churn", axis=1)
y = df["churn"]

# Convert categorical columns into numeric columns
X = pd.get_dummies(X, drop_first=True)

X.head()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# ============================================================
# Scaling for KNN
# ============================================================

# KNN uses distance between records.
# If one column has very large values, it can dominate the distance calculation.
# Example:
# income may be 100000, but age may be 30.
# So we scale values before applying KNN.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

In [ ]:
# Build and train KNN model
model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train_scaled, y_train)

print("KNN model trained successfully.")

In [ ]:
# Predict
y_pred = model.predict(X_test_scaled)

print("First 10 predictions:", y_pred[:10])

In [ ]:
# Evaluation metrics

# Accuracy: total correct predictions out of all predictions.
accuracy = accuracy_score(y_test, y_pred)

# Precision: out of predicted churn customers, how many were actually churn.
precision = precision_score(y_test, y_pred)

# Recall: out of actual churn customers, how many were correctly found.
recall = recall_score(y_test, y_pred)

# F1 Score: balance between precision and recall.
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Save both scaler and model using pickle
# We save scaler also because future data must be scaled in the same way.

knn_package = {
    "scaler": scaler,
    "model": model
}

model_path = "../models/knn_churn_with_scaler.pkl"

with open(model_path, "wb") as file:
    pickle.dump(knn_package, file)

print("KNN model and scaler saved successfully at:", model_path)

In [ ]:
# Load pickle and test
with open("../models/knn_churn_with_scaler.pkl", "rb") as file:
    loaded_package = pickle.load(file)

loaded_scaler = loaded_package["scaler"]
loaded_model = loaded_package["model"]

loaded_predictions = loaded_model.predict(loaded_scaler.transform(X_test))

print("Loaded KNN accuracy:", accuracy_score(y_test, loaded_predictions))